# 04d. Avaliar faixa `[0,05, 0,25)`

Diagnóstico do buraco `com_bloco_p_lt_025` do [`04c_decompor_sem_cpf.ipynb`](04c_decompor_sem_cpf.ipynb):
Censo sem CPF cujo melhor par no parquet está em `0,05 ≤ p < 0,25`.

Exige [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb) com
`threshold_match_probability = 0,05` (senão a faixa vem vazia). Não muda a
escada do [`04_atribuir.ipynb`](04_atribuir.ipynb). Níveis de nome/data na
mesma ordem do treino (`02`).


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    SPLINK_ATRIBUICAO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO = 0.05
TETO = 0.25

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (02b com predict 0,05)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print('Faixa:', PISO, '<= p <', TETO)


In [ ]:
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability,
    match_weight
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao AS
SELECT unique_id_censo, unique_id_cpf, match_probability, degrau, regra
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]
n_atrib = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
p_min = con.execute('SELECT MIN(match_probability) FROM splink_predictions').fetchone()[0]
print('Pares no parquet:', f'{n_pred:,}')
print('Atribuições:', f'{n_atrib:,}')
print('MIN(p) no parquet:', p_min)
if p_min is not None and p_min >= PISO + 1e-9:
    print(
        'AVISO: MIN(p) >=', PISO,
        '— rode o 02b com threshold_match_probability =', PISO,
    )


Sem CPF no 04; melhor par na faixa `[0,05, 0,25)`. Conta também quem continua
sem par ≥ 0,05 (ainda abaixo do predict).


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
    SELECT 1 FROM atribuicao a
    WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_faixa AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    p.match_weight
FROM splink_predictions p
JOIN censo_sem_cpf s ON s.unique_id = p.unique_id_censo
WHERE p.match_probability >= {PISO}
  AND p.match_probability < {TETO}
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY p.match_probability DESC, p.unique_id_cpf
) = 1
''')

display(con.execute(f'''
SELECT
    n_sem_cpf,
    n_com_faixa,
    n_sem_cpf - n_com_faixa AS n_sem_p005,
    ROUND(100.0 * n_com_faixa / NULLIF(n_sem_cpf, 0), 1) AS pct_com_faixa,
    ROUND(100.0 * (n_sem_cpf - n_com_faixa) / NULLIF(n_sem_cpf, 0), 1)
        AS pct_sem_p005
FROM (
    SELECT
        (SELECT COUNT(*) FROM censo_sem_cpf) AS n_sem_cpf,
        (SELECT COUNT(*) FROM melhor_faixa) AS n_com_faixa
)
''').df())


Níveis na ordem do treino: nome (Exact → Prefixo → DL≤1 → tokens → JW → Else)
e data (Null → Exact → troca → dia → mês → ano±1 → DL≤1 → DL≤2 → Else).


In [ ]:
split_c = "string_split(ca.nome_completo_phon, ' ')"
split_p = "string_split(pb.nome_completo_phon, ' ')"
n_min = f'least(len({split_c}), len({split_p}))'
prefixo_sql = (
    f'{n_min} >= 2 AND len({split_c}) <> len({split_p}) AND '
    f'list_slice({split_c}, 1, {n_min}) = list_slice({split_p}, 1, {n_min})'
)
jw_ultimo_095_sql = (
    f"{split_c}[-1] = {split_p}[-1] AND "
    'jaro_winkler_similarity(ca.nome_completo_phon, pb.nome_completo_phon) >= 0.95'
)
jw_ultimo_092_sql = (
    f"{split_c}[-1] = {split_p}[-1] AND "
    'jaro_winkler_similarity(ca.nome_completo_phon, pb.nome_completo_phon) >= 0.92'
)
dl_completo_1_sql = (
    'damerau_levenshtein(ca.nome_completo_phon, pb.nome_completo_phon) <= 1'
)
n_tok = f'len({split_c})'
mesmo_n = f'{n_tok} = len({split_p}) AND {n_tok} >= 3'
zip_tok = f'list_zip({split_c}, {split_p})'
dif_count = (
    'list_aggregate(list_transform('
    f'{zip_tok}, '
    "x -> CASE WHEN x[1] <> x[2] THEN 1 ELSE 0 END"
    "), 'sum')"
)
dif_max_dl = (
    'list_aggregate(list_transform('
    f'{zip_tok}, '
    "x -> CASE WHEN x[1] <> x[2] "
    "THEN damerau_levenshtein(x[1], x[2]) ELSE 0 END"
    "), 'max')"
)
um_token_sql = (
    f'{mesmo_n} AND {split_c}[-1] = {split_p}[-1] '
    f'AND {dif_count} = 1 AND {dif_max_dl} <= 2'
)
dois_tokens_sql = (
    f'{mesmo_n} AND {dif_count} BETWEEN 1 AND 2 AND {dif_max_dl} <= 2'
)
n_c = f'len({split_c})'
n_p = f'len({split_p})'
ordem_sql = (
    f'{n_c} = {n_p} AND {n_c} >= 3 '
    f'AND list_sort({split_c}) = list_sort({split_p}) '
    f'AND {split_c} <> {split_p}'
)

ors_token = []
for k in range(1, 11):
    if k == 1:
        sem_c = f'list_slice({split_c}, 2, {n_c})'
        sem_p = f'list_slice({split_p}, 2, {n_p})'
    else:
        sem_c = (
            f'list_concat(list_slice({split_c}, 1, {k - 1}), '
            f'list_slice({split_c}, {k + 1}, {n_c}))'
        )
        sem_p = (
            f'list_concat(list_slice({split_p}, 1, {k - 1}), '
            f'list_slice({split_p}, {k + 1}, {n_p}))'
        )
    ors_token.append(f'({n_c} >= {k} AND {sem_c} = {split_p})')
    ors_token.append(f'({n_p} >= {k} AND {sem_p} = {split_c})')
token_a_mais_sql = (
    f'abs({n_c} - {n_p}) = 1 AND least({n_c}, {n_p}) >= 3 AND ('
    + ' OR '.join(ors_token)
    + ')'
)

null_data_sql = (
    '(ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL) AND '
    '(ca.mes_nascimento IS NULL OR pb.mes_nascimento IS NULL OR '
    'ca.dia_nascimento IS NULL OR pb.dia_nascimento IS NULL)'
)
troca_mes_dia_sql = (
    'ca.ano_nascimento = pb.ano_nascimento AND '
    'ca.mes_nascimento = pb.dia_nascimento AND '
    'ca.dia_nascimento = pb.mes_nascimento AND '
    'ca.mes_nascimento <> ca.dia_nascimento'
)
dia_diferente_sql = (
    'ca.ano_nascimento = pb.ano_nascimento AND '
    'ca.mes_nascimento = pb.mes_nascimento AND '
    'ca.dia_nascimento <> pb.dia_nascimento'
)
mes_diferente_sql = (
    'ca.ano_nascimento = pb.ano_nascimento AND '
    'ca.dia_nascimento = pb.dia_nascimento AND '
    'ca.mes_nascimento <> pb.mes_nascimento'
)
ano_pm1_sql = (
    'ca.mes_nascimento = pb.mes_nascimento AND '
    'ca.dia_nascimento = pb.dia_nascimento AND '
    'abs(try_cast(ca.ano_nascimento AS INTEGER) - '
    'try_cast(pb.ano_nascimento AS INTEGER)) = 1'
)

con.execute(f'''
CREATE OR REPLACE TABLE faixa_niveis AS
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    m.match_weight,
    ca.nome_completo_phon AS nome_censo,
    pb.nome_completo_phon AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    CASE
        WHEN ca.nome_completo_phon IS NULL OR pb.nome_completo_phon IS NULL
            THEN 'null'
        WHEN ca.nome_completo_phon = pb.nome_completo_phon
            THEN 'exato'
        WHEN {prefixo_sql}
            THEN 'prefixo'
        WHEN {dl_completo_1_sql}
            THEN 'dl_1'
        WHEN {um_token_sql}
            THEN 'um_token_dl2'
        WHEN {dois_tokens_sql}
            THEN 'dois_tokens_dl2'
        WHEN {token_a_mais_sql}
            THEN 'token_a_mais'
        WHEN {ordem_sql}
            THEN 'ordem'
        WHEN {jw_ultimo_095_sql}
            THEN 'jw_095'
        WHEN {jw_ultimo_092_sql}
            THEN 'jw_092'
        ELSE 'else'
    END AS nivel_nome,
    CASE
        WHEN {null_data_sql}
            THEN 'null'
        WHEN ca.data_nascimento = pb.data_nascimento
            THEN 'exato'
        WHEN {troca_mes_dia_sql}
            THEN 'mes_dia_trocados'
        WHEN {dia_diferente_sql}
            THEN 'dia_diferente'
        WHEN {mes_diferente_sql}
            THEN 'mes_diferente'
        WHEN {ano_pm1_sql}
            THEN 'ano_pm1'
        WHEN damerau_levenshtein(ca.data_nascimento, pb.data_nascimento) <= 1
            THEN 'dl_1'
        WHEN damerau_levenshtein(ca.data_nascimento, pb.data_nascimento) <= 2
            THEN 'dl_2'
        ELSE 'else'
    END AS nivel_data
FROM melhor_faixa m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
''')
print('Pares classificados:', con.execute('SELECT COUNT(*) FROM faixa_niveis').fetchone()[0])


In [ ]:
display(con.execute('''
SELECT
    nivel_nome,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(match_weight), 2) AS avg_w
FROM faixa_niveis
GROUP BY 1
ORDER BY n DESC
''').df())


In [ ]:
display(con.execute('''
SELECT
    nivel_data,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(match_weight), 2) AS avg_w
FROM faixa_niveis
GROUP BY 1
ORDER BY n DESC
''').df())


In [ ]:
display(con.execute('''
SELECT
    nivel_nome,
    nivel_data,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p
FROM faixa_niveis
GROUP BY 1, 2
ORDER BY n DESC
''').df())


Amostra dos cruzamentos mais frequentes (nome ELSE, data null, exato+null, etc.).


In [ ]:
display(con.execute('''
SELECT
    match_probability,
    nivel_nome,
    nivel_data,
    nome_censo,
    nome_cpf,
    dob_censo,
    dob_cpf
FROM faixa_niveis
WHERE nivel_nome = 'else'
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute('''
SELECT
    match_probability,
    nivel_nome,
    nivel_data,
    nome_censo,
    nome_cpf,
    dob_censo,
    dob_cpf
FROM faixa_niveis
WHERE nivel_nome = 'exato' AND nivel_data = 'null'
ORDER BY match_probability DESC
LIMIT 40
''').df())
